<a href="https://colab.research.google.com/github/Ayushmaantiwari/motorsim/blob/main/MotorSim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1: Install/import required packages
%pip -q install ipywidgets scipy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output, FileLink
from scipy import signal
from scipy.optimize import brentq

print("Libraries loaded. Continue to the GUI cell.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 19.4 MB/s eta 0:00:00
Libraries loaded. Continue to the GUI cell.


In [3]:
# Cell 2: Build the voltage-input DC motor GUI
# Input: armature voltage Va (V)
# Outputs: armature current Ia (A), angular velocity omega (rad/s), position theta (rad)

# ---------- Motor parameter controls ----------
Ra_w = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description="Ra (ohm)", readout_format=".2f", continuous_update=False, style={"description_width":"100px"}, layout=widgets.Layout(width="420px"))
La_w = widgets.FloatSlider(value=0.5, min=0.01, max=2.0, step=0.01, description="La (H)", readout_format=".2f", continuous_update=False, style={"description_width":"100px"}, layout=widgets.Layout(width="420px"))
J_w = widgets.FloatSlider(value=0.01, min=0.001, max=0.2, step=0.001, description="J (kg m²)", readout_format=".3f", continuous_update=False, style={"description_width":"100px"}, layout=widgets.Layout(width="420px"))
b_w = widgets.FloatSlider(value=0.1, min=0.001, max=1.0, step=0.001, description="b (N m s)", readout_format=".3f", continuous_update=False, style={"description_width":"100px"}, layout=widgets.Layout(width="420px"))
Kt_w = widgets.FloatSlider(value=0.01, min=0.001, max=1.0, step=0.001, description="Kt (N m/A)", readout_format=".3f", continuous_update=False, style={"description_width":"100px"}, layout=widgets.Layout(width="420px"))
Ke_w = widgets.FloatSlider(value=0.01, min=0.001, max=1.0, step=0.001, description="Ke (V s/rad)", readout_format=".3f", continuous_update=False, style={"description_width":"100px"}, layout=widgets.Layout(width="420px"))

# ---------- Voltage input and simulation controls ----------
voltage_w = widgets.FloatSlider(value=12.0, min=-48.0, max=48.0, step=0.5, description="Armature voltage (V)", readout_format=".1f", continuous_update=False, style={"description_width":"150px"}, layout=widgets.Layout(width="500px"))
duration_w = widgets.FloatSlider(value=10.0, min=1.0, max=30.0, step=1.0, description="Duration (s)", readout_format=".0f", continuous_update=False, style={"description_width":"150px"}, layout=widgets.Layout(width="420px"))
points_w = widgets.IntSlider(value=3000, min=500, max=8000, step=500, description="Samples", continuous_update=False, style={"description_width":"150px"}, layout=widgets.Layout(width="420px"))
load_w = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.01, description="Load torque (N m)", readout_format=".2f", continuous_update=False, style={"description_width":"150px"}, layout=widgets.Layout(width="420px"))

run_button = widgets.Button(description="Run simulation", button_style="success", icon="play", layout=widgets.Layout(width="200px", height="42px"))
reset_button = widgets.Button(description="Reset defaults", button_style="", icon="refresh", layout=widgets.Layout(width="200px", height="42px"))
download_box = widgets.Output()
results_output = widgets.Output()
plot_output = widgets.Output()
status_output = widgets.Output()

def motor_state_equations(t, state, voltage, Ra, La, J, b, Kt, Ke, load_torque):
    current, omega, theta = state
    di_dt = (voltage - Ra * current - Ke * omega) / La
    domega_dt = (Kt * current - b * omega - load_torque) / J
    dtheta_dt = omega
    return [di_dt, domega_dt, dtheta_dt]

def run_simulation(_=None):
    for out in [plot_output, results_output, status_output, download_box]:
        with out:
            clear_output(wait=True)

    Ra, La, J, b, Kt, Ke = [w.value for w in [Ra_w, La_w, J_w, b_w, Kt_w, Ke_w]]
    voltage = float(voltage_w.value)
    duration = float(duration_w.value)
    n_samples = int(points_w.value)
    load_torque = float(load_w.value)
    t = np.linspace(0, duration, n_samples)

    try:
        from scipy.integrate import solve_ivp
        result = solve_ivp(
            fun=lambda time, state: motor_state_equations(time, state, voltage, Ra, La, J, b, Kt, Ke, load_torque),
            t_span=(0.0, duration), y0=[0.0, 0.0, 0.0], t_eval=t,
            method="RK45", rtol=1e-7, atol=1e-9
        )
        if not result.success:
            raise RuntimeError(result.message)

        current, omega, theta = result.y
        df = pd.DataFrame({
            "Time (s)": result.t,
            "Armature voltage (V)": np.full_like(result.t, voltage),
            "Armature current (A)": current,
            "Angular velocity (rad/s)": omega,
            "Angular position (rad)": theta
        })

        with plot_output:
            fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
            axes[0].plot(result.t, current, linewidth=2)
            axes[0].set_title("Armature Current")
            axes[0].set_ylabel("Current (A)")
            axes[0].grid(True, alpha=0.3)

            axes[1].plot(result.t, omega, linewidth=2)
            axes[1].set_title("Angular Velocity")
            axes[1].set_ylabel("Angular velocity (rad/s)")
            axes[1].grid(True, alpha=0.3)

            axes[2].plot(result.t, theta, linewidth=2)
            axes[2].set_title("Angular Position")
            axes[2].set_xlabel("Time (s)")
            axes[2].set_ylabel("Position (rad)")
            axes[2].grid(True, alpha=0.3)

            fig.suptitle(f"DC Motor Response to {voltage:.2f} V Input")
            plt.tight_layout()
            fig.savefig("dc_motor_voltage_response.png", dpi=200, bbox_inches="tight")
            plt.show()

        with results_output:
            print("Motor transfer function:")
            print(f"Omega(s)/Va(s) = {Kt:.5g} / ({La*J:.5g}s^2 + {(La*b + Ra*J):.5g}s + {(Ra*b + Kt*Ke):.5g})")
            print("\nSimulation settings:")
            print(f"Armature voltage: {voltage:.3f} V")
            print(f"Load torque: {load_torque:.3f} N m")
            print(f"Duration: {duration:.2f} s")
            print("\nFinal simulated values:")
            print(f"Armature current: {current[-1]:.6g} A")
            print(f"Angular velocity: {omega[-1]:.6g} rad/s")
            print(f"Angular position: {theta[-1]:.6g} rad")
            display(df.tail(10).round(5))

        df.to_csv("dc_motor_voltage_outputs.csv", index=False)
        with open("dc_motor_voltage_simulation_report.txt", "w", encoding="utf-8") as f:
            f.write("DC MOTOR VOLTAGE-INPUT SIMULATION REPORT\n")
            f.write("========================================\n\n")
            f.write(f"Voltage: {voltage:.6g} V\nLoad torque: {load_torque:.6g} N m\n")
            f.write(f"Ra={Ra}, La={La}, J={J}, b={b}, Kt={Kt}, Ke={Ke}\n\n")
            f.write("Motor transfer function:\n")
            f.write(f"Omega(s)/Va(s) = {Kt} / ({La*J}s^2 + {(La*b + Ra*J)}s + {(Ra*b + Kt*Ke)})\n\n")
            f.write(f"Final current: {current[-1]} A\nFinal angular velocity: {omega[-1]} rad/s\nFinal angular position: {theta[-1]} rad\n\n")
            f.write("di/dt = (Va - Ra*i - Ke*omega)/La\n")
            f.write("domega/dt = (Kt*i - b*omega - load_torque)/J\n")
            f.write("dtheta/dt = omega\n")
            f.write("\nAssumptions: constant voltage, fixed parameters, initial rest state, no voltage/current saturation.\n")

        with download_box:
            print("Files generated:")
            display(FileLink("dc_motor_voltage_outputs.csv"))
            display(FileLink("dc_motor_voltage_response.png"))
            display(FileLink("dc_motor_voltage_simulation_report.txt"))

    except Exception as exc:
        with status_output:
            print(f"Simulation error: {exc}")

def reset_defaults(_):
    Ra_w.value, La_w.value, J_w.value = 1.0, 0.5, 0.01
    b_w.value, Kt_w.value, Ke_w.value = 0.1, 0.01, 0.01
    voltage_w.value, duration_w.value, points_w.value, load_w.value = 12.0, 10.0, 3000, 0.0

run_button.on_click(run_simulation)
reset_button.on_click(reset_defaults)

motor_box = widgets.VBox([
    widgets.HTML("<h3>1. DC motor parameters</h3>"),
    widgets.HBox([widgets.VBox([Ra_w, La_w, J_w]), widgets.VBox([b_w, Kt_w, Ke_w])])
])
input_box = widgets.VBox([
    widgets.HTML("<h3>2. Voltage input and simulation settings</h3>"),
    voltage_w,
    widgets.HBox([duration_w, points_w, load_w])
])

display(widgets.VBox([
    motor_box, input_box, widgets.HBox([run_button, reset_button]),
    status_output,
    widgets.HTML("<h3>3. Motor output plots</h3>"), plot_output,
    widgets.HTML("<h3>4. Simulation results</h3>"), results_output,
    widgets.HTML("<h3>5. Export</h3>"), download_box
]))


In [4]:
# Cell 2: Build the GUI and simulation engine

# ---------- Motor parameter controls ----------
Ra_w = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1,
                           description="Ra (ohm)", readout_format=".2f",
                           continuous_update=False, style={"description_width":"100px"},
                           layout=widgets.Layout(width="420px"))
La_w = widgets.FloatSlider(value=0.5, min=0.01, max=2.0, step=0.01,
                           description="La (H)", readout_format=".2f",
                           continuous_update=False, style={"description_width":"100px"},
                           layout=widgets.Layout(width="420px"))
J_w = widgets.FloatSlider(value=0.01, min=0.001, max=0.2, step=0.001,
                          description="J (kg m²)", readout_format=".3f",
                          continuous_update=False, style={"description_width":"100px"},
                          layout=widgets.Layout(width="420px"))
b_w = widgets.FloatSlider(value=0.1, min=0.001, max=1.0, step=0.001,
                          description="b (N m s)", readout_format=".3f",
                          continuous_update=False, style={"description_width":"100px"},
                          layout=widgets.Layout(width="420px"))
Kt_w = widgets.FloatSlider(value=0.01, min=0.001, max=1.0, step=0.001,
                           description="Kt (N m/A)", readout_format=".3f",
                           continuous_update=False, style={"description_width":"100px"},
                           layout=widgets.Layout(width="420px"))
Ke_w = widgets.FloatSlider(value=0.01, min=0.001, max=1.0, step=0.001,
                           description="Ke (V s/rad)", readout_format=".3f",
                           continuous_update=False, style={"description_width":"100px"},
                           layout=widgets.Layout(width="420px"))

# ---------- Reference and simulation controls ----------
ref_w = widgets.FloatSlider(value=1.0, min=0.1, max=100.0, step=0.1,
                            description="Reference (rad/s)", readout_format=".1f",
                            continuous_update=False, style={"description_width":"150px"},
                            layout=widgets.Layout(width="420px"))
duration_w = widgets.FloatSlider(value=10.0, min=1.0, max=30.0, step=1.0,
                                 description="Duration (s)", readout_format=".0f",
                                 continuous_update=False, style={"description_width":"150px"},
                                 layout=widgets.Layout(width="420px"))
points_w = widgets.IntSlider(value=3000, min=500, max=8000, step=500,
                             description="Samples", continuous_update=False,
                             style={"description_width":"150px"},
                             layout=widgets.Layout(width="420px"))

# ---------- Controller gains ----------
gain_defaults = {
    "P":   (100.0, 0.0,   0.0),
    "PD":  (100.0, 0.0,   5.0),
    "PI":  (100.0, 200.0, 0.0),
    "PID": (100.0, 200.0, 5.0),
}
gain_widgets = {}
enable_widgets = {}

for name, (kp0, ki0, kd0) in gain_defaults.items():
    enable_widgets[name] = widgets.Checkbox(
        value=True, description=f"Enable {name}", indent=False,
        layout=widgets.Layout(width="150px")
    )
    kp = widgets.FloatText(value=kp0, description="Kp",
                           style={"description_width":"35px"},
                           layout=widgets.Layout(width="145px"))
    ki = widgets.FloatText(value=ki0, description="Ki",
                           style={"description_width":"35px"},
                           layout=widgets.Layout(width="145px"))
    kd = widgets.FloatText(value=kd0, description="Kd",
                           style={"description_width":"35px"},
                           layout=widgets.Layout(width="145px"))
    gain_widgets[name] = {"Kp": kp, "Ki": ki, "Kd": kd}

gain_rows = []
for name in ["P", "PD", "PI", "PID"]:
    gain_rows.append(widgets.HBox([
        enable_widgets[name],
        gain_widgets[name]["Kp"],
        gain_widgets[name]["Ki"],
        gain_widgets[name]["Kd"],
    ], layout=widgets.Layout(align_items="center")))

run_button = widgets.Button(
    description="Run simulation", button_style="success",
    icon="play", layout=widgets.Layout(width="200px", height="42px")
)
reset_button = widgets.Button(
    description="Reset default gains", button_style="",
    icon="refresh", layout=widgets.Layout(width="200px", height="42px")
)
download_box = widgets.Output()
results_output = widgets.Output()
plot_output = widgets.Output()
status_output = widgets.Output()

def controller_tf(name, kp, ki, kd):
    """Create controller transfer function C(s)."""
    if name == "P":
        return signal.TransferFunction([kp], [1.0])
    if name == "PD":
        return signal.TransferFunction([kd, kp], [1.0])
    if name == "PI":
        return signal.TransferFunction([kp, ki], [1.0, 0.0])
    # PID: (Kd*s^2 + Kp*s + Ki) / s
    return signal.TransferFunction([kd, kp, ki], [1.0, 0.0])

def poly_add_aligned(a, b):
    """Add descending-power polynomial coefficient arrays."""
    n = max(len(a), len(b))
    aa = np.pad(np.asarray(a, dtype=float), (n-len(a), 0))
    bb = np.pad(np.asarray(b, dtype=float), (n-len(b), 0))
    return aa + bb

def make_closed_loop(name, kp, ki, kd, num_motor, den_motor):
    """Closed-loop T(s)=C(s)G(s)/(1+C(s)G(s)) for unity feedback."""
    C = controller_tf(name, kp, ki, kd)
    num_ol = np.polymul(C.num, num_motor)
    den_ol = np.polymul(C.den, den_motor)
    den_cl = poly_add_aligned(den_ol, num_ol)
    return signal.TransferFunction(num_ol, den_cl)

def calculate_metrics(t, y, reference, stable):
    """Calculate response metrics, with settling time relative to final value."""
    y = np.asarray(y, dtype=float)
    if not np.all(np.isfinite(y)):
        return {"Rise time (s)": np.nan, "Settling time (s)": np.nan,
                "Overshoot (%)": np.nan, "Final speed (rad/s)": np.nan,
                "Steady-state error (rad/s)": np.nan}

    # For stable systems, use final-value theorem (DC gain) where possible.
    if stable:
        # The response simulation endpoint is a robust fallback.
        y_final = float(y[-1])
    else:
        y_final = float(y[-1])

    error = abs(reference - y_final)

    if abs(y_final) > 1e-10:
        lo, hi = 0.1*y_final, 0.9*y_final
        if y_final >= 0:
            i10 = np.where(y >= lo)[0]
            i90 = np.where(y >= hi)[0]
        else:
            i10 = np.where(y <= lo)[0]
            i90 = np.where(y <= hi)[0]
        rise = np.nan
        if len(i10) and len(i90):
            start = i10[0]
            later = i90[i90 >= start]
            if len(later):
                rise = float(t[later[0]] - t[start])

        overshoot = max(0.0, (np.max(y)-y_final)/abs(y_final)*100.0)
        band = 0.02 * max(abs(y_final), 1e-10)
        outside = np.where(np.abs(y-y_final) > band)[0]
        settling = float(t[outside[-1]+1]) if len(outside) and outside[-1] < len(t)-1 else (0.0 if not len(outside) else np.nan)
    else:
        rise, overshoot, settling = np.nan, np.nan, np.nan

    # An unstable response should not be presented as having a meaningful
    # steady-state tracking error.
    if not stable:
        error = np.nan
        settling = np.nan

    return {"Rise time (s)": rise, "Settling time (s)": settling,
            "Overshoot (%)": overshoot, "Final speed (rad/s)": y_final,
            "Steady-state error (rad/s)": error}

def run_simulation(_=None):
    with plot_output:
        clear_output(wait=True)
    with results_output:
        clear_output(wait=True)
    with status_output:
        clear_output(wait=True)
    with download_box:
        clear_output(wait=True)

    # Validate input gains
    selected = [name for name in ["P", "PD", "PI", "PID"] if enable_widgets[name].value]
    if not selected:
        with status_output:
            print("Select at least one controller.")
        return

    all_gains = {}
    for name in selected:
        vals = {key: float(gain_widgets[name][key].value) for key in ["Kp", "Ki", "Kd"]}
        if not all(np.isfinite(list(vals.values()))):
            with status_output:
                print(f"{name}: gain values must be finite numbers.")
            return
        all_gains[name] = vals

    # Motor model
    Ra, La, J, b, Kt, Ke = [w.value for w in [Ra_w, La_w, J_w, b_w, Kt_w, Ke_w]]
    num_motor = [Kt]
    den_motor = [La*J, La*b + Ra*J, Ra*b + Kt*Ke]

    reference = float(ref_w.value)
    t = np.linspace(0, float(duration_w.value), int(points_w.value))
    all_y, rows, systems = {}, [], {}

    for name, gains in all_gains.items():
        try:
            sys_cl = make_closed_loop(name, gains["Kp"], gains["Ki"], gains["Kd"],
                                      num_motor, den_motor)
            systems[name] = sys_cl
            poles = np.roots(sys_cl.den)
            stable = bool(np.all(np.real(poles) < -1e-9))
            tout, y = signal.step(sys_cl, T=t)
            y = np.asarray(y).squeeze() * reference
            all_y[name] = y

            # Use DC gain for final value when stable and finite; then recalculate
            # response metrics against that estimated final value.
            if stable:
                dc_num = np.polyval(sys_cl.num, 0.0)
                dc_den = np.polyval(sys_cl.den, 0.0)
                if abs(dc_den) > 1e-12:
                    final_value = float(dc_num / dc_den * reference)
                    # Use simulated y for transient metrics, but use final value
                    # from DC gain for steady-state error.
                    m = calculate_metrics(t, y, reference, stable)
                    m["Final speed (rad/s)"] = final_value
                    m["Steady-state error (rad/s)"] = abs(reference-final_value)
                else:
                    m = calculate_metrics(t, y, reference, stable)
            else:
                m = calculate_metrics(t, y, reference, stable)

            rows.append({
                "Controller": name,
                "Kp": gains["Kp"], "Ki": gains["Ki"], "Kd": gains["Kd"],
                "Stable": stable,
                **m
            })
        except Exception as exc:
            rows.append({
                "Controller": name,
                "Kp": gains["Kp"], "Ki": gains["Ki"], "Kd": gains["Kd"],
                "Stable": False,
                "Rise time (s)": np.nan, "Settling time (s)": np.nan,
                "Overshoot (%)": np.nan, "Final speed (rad/s)": np.nan,
                "Steady-state error (rad/s)": np.nan,
                "Error": str(exc)
            })

    df = pd.DataFrame(rows)

    # Plot all responses and a small dashboard of metrics.
    with plot_output:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        for name, y in all_y.items():
            axes[0].plot(t, y, linewidth=2, label=name)
        axes[0].axhline(reference, color="black", linestyle="--", label="Reference")
        axes[0].set_title("Closed-loop speed responses")
        axes[0].set_xlabel("Time (s)")
        axes[0].set_ylabel("Angular speed (rad/s)")
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()

        metric_names = ["Rise time (s)", "Settling time (s)", "Overshoot (%)",
                        "Steady-state error (rad/s)"]
        # Show settling time and other metrics in separate figure for readable scales.
        plt.tight_layout()
        plt.show()

        fig, axs = plt.subplots(2, 2, figsize=(12, 8))
        for ax, metric in zip(axs.ravel(), metric_names):
            vals = df.set_index("Controller")[metric]
            ax.bar(vals.index.astype(str), vals.values)
            ax.set_title(metric)
            ax.grid(axis="y", alpha=0.3)
            ax.tick_params(axis="x", rotation=15)
        fig.suptitle("Performance metric comparison")
        plt.tight_layout()
        plt.show()

        # Save figures for download
        fig1 = plt.figure(figsize=(9, 5))
        for name, y in all_y.items():
            plt.plot(t, y, linewidth=2, label=name)
        plt.axhline(reference, color="black", linestyle="--", label="Reference")
        plt.title("DC Motor Speed Control — Step Response")
        plt.xlabel("Time (s)")
        plt.ylabel("Angular speed (rad/s)")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        fig1.savefig("dc_motor_step_responses.png", dpi=200, bbox_inches="tight")
        plt.close(fig1)

    with results_output:
        display(df.round(5))
        print("\nMotor transfer function:")
        print(f"G(s) = {Kt:.5g} / ({den_motor[0]:.5g}s^2 + {den_motor[1]:.5g}s + {den_motor[2]:.5g})")

    # Save CSV and simple report
    df.to_csv("dc_motor_controller_metrics.csv", index=False)
    with open("dc_motor_simulation_report.txt", "w", encoding="utf-8") as f:
        f.write("DC MOTOR SPEED CONTROL SIMULATION REPORT\n")
        f.write("========================================\n\n")
        f.write(f"Motor transfer function: G(s) = {Kt} / ({den_motor[0]}s^2 + {den_motor[1]}s + {den_motor[2]})\n")
        f.write(f"Reference speed: {reference} rad/s\n")
        f.write(f"Simulation duration: {t[-1]} s\n\n")
        f.write("Controller gains and performance:\n")
        f.write(df.round(5).to_string(index=False))
        f.write("\n\nInterpretation:\n")
        f.write("P control is simple but usually leaves steady-state error. PD can improve damping but does not independently remove steady-state error. PI adds integral action and can remove steady-state error for a stable system tracking a constant reference. PID combines all three actions and may offer a good compromise after tuning. Compare the actual metrics and stability before selecting a controller.\n")
        f.write("\nLimitations: linear motor model, unity feedback, no load disturbance, actuator saturation, derivative filtering, or integral anti-windup.\n")

    with download_box:
        print("Files created in the Colab working directory:")
        display(FileLink("dc_motor_controller_metrics.csv"))
        display(FileLink("dc_motor_step_responses.png"))
        display(FileLink("dc_motor_simulation_report.txt"))

def reset_gains(_):
    for name, defaults in gain_defaults.items():
        gain_widgets[name]["Kp"].value = defaults[0]
        gain_widgets[name]["Ki"].value = defaults[1]
        gain_widgets[name]["Kd"].value = defaults[2]
        enable_widgets[name].value = True

run_button.on_click(run_simulation)
reset_button.on_click(reset_gains)

motor_box = widgets.VBox([
    widgets.HTML("<h3>1. DC motor parameters</h3>"),
    widgets.HBox([widgets.VBox([Ra_w, La_w, J_w]), widgets.VBox([b_w, Kt_w, Ke_w])])
])
sim_box = widgets.VBox([
    widgets.HTML("<h3>2. Reference and simulation settings</h3>"),
    widgets.HBox([ref_w, duration_w, points_w])
])
controller_box = widgets.VBox([
    widgets.HTML("<h3>3. Controller selection and gains</h3>"),
    widgets.HTML("<small>Unchecked controllers are excluded. Set unused gains to zero for P/PD/PI.</small>"),
    *gain_rows
])

display(widgets.VBox([
    motor_box, sim_box, controller_box,
    widgets.HBox([run_button, reset_button]),
    status_output,
    widgets.HTML("<h3>4. Plots</h3>"), plot_output,
    widgets.HTML("<h3>5. Performance results</h3>"), results_output,
    widgets.HTML("<h3>6. Export</h3>"), download_box
]))
